In [1]:
import os, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2), # Adds robustness
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 100% Data Loader
data_dir = '../415-Final-Project/ecse-415-winter-2026-dog-vs-cat-classification/train/train'
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, num_workers=2)

print("\n--- Training EfficientNetV2-S ---")
model_eff = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
# EfficientNet's final layer is inside a classifier module
model_eff.classifier[1] = nn.Linear(model_eff.classifier[1].in_features, 2)
model_eff = model_eff.to(device)

optimizer = optim.Adam(model_eff.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model_eff.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_eff(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i+1) % 100 == 0: print(f"  Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

torch.save(model_eff.state_dict(), 'efficientnet_best.pth')
print("EfficientNet Saved!")


Using device: cuda

--- Training EfficientNetV2-S ---
  Batch 100/625 | Loss: 0.0502
  Batch 200/625 | Loss: 0.0730
  Batch 300/625 | Loss: 0.3445
  Batch 400/625 | Loss: 0.0781
  Batch 500/625 | Loss: 0.0298
  Batch 600/625 | Loss: 0.0715
  Batch 100/625 | Loss: 0.0183
  Batch 200/625 | Loss: 0.1316
  Batch 300/625 | Loss: 0.0073
  Batch 400/625 | Loss: 0.0429
  Batch 500/625 | Loss: 0.0070
  Batch 600/625 | Loss: 0.0937
  Batch 100/625 | Loss: 0.1181
  Batch 200/625 | Loss: 0.0394
  Batch 300/625 | Loss: 0.0369
  Batch 400/625 | Loss: 0.0313
  Batch 500/625 | Loss: 0.0804
  Batch 600/625 | Loss: 0.0644
  Batch 100/625 | Loss: 0.1138
  Batch 200/625 | Loss: 0.0815
  Batch 300/625 | Loss: 0.0064
  Batch 400/625 | Loss: 0.0114
  Batch 500/625 | Loss: 0.0563
  Batch 600/625 | Loss: 0.0428
  Batch 100/625 | Loss: 0.0142
  Batch 200/625 | Loss: 0.0130
  Batch 300/625 | Loss: 0.0425
  Batch 400/625 | Loss: 0.0703
  Batch 500/625 | Loss: 0.1131
  Batch 600/625 | Loss: 0.0066
EfficientNet Sav

In [2]:
print("\nTraining ConvNeXt-Tiny")
model_conv = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
# ConvNeXt final layer
model_conv.classifier[2] = nn.Linear(model_conv.classifier[2].in_features, 2)
model_conv = model_conv.to(device)

optimizer = optim.Adam(model_conv.parameters(), lr=0.0001)

for epoch in range(5):
    model_conv.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_conv(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i+1) % 100 == 0: print(f"Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

torch.save(model_conv.state_dict(), 'convnext_best.pth')
print("ConvNeXt Saved!")



Training ConvNeXt-Tiny
Batch 100/625 | Loss: 0.0167
Batch 200/625 | Loss: 0.0662
Batch 300/625 | Loss: 0.1411
Batch 400/625 | Loss: 0.0658
Batch 500/625 | Loss: 0.0572
Batch 600/625 | Loss: 0.0133
Batch 100/625 | Loss: 0.0186
Batch 200/625 | Loss: 0.0387
Batch 300/625 | Loss: 0.0299
Batch 400/625 | Loss: 0.1072
Batch 500/625 | Loss: 0.2635
Batch 600/625 | Loss: 0.0736
Batch 100/625 | Loss: 0.0679
Batch 200/625 | Loss: 0.0068
Batch 300/625 | Loss: 0.0077
Batch 400/625 | Loss: 0.0182
Batch 500/625 | Loss: 0.0466
Batch 600/625 | Loss: 0.0808
Batch 100/625 | Loss: 0.1099
Batch 200/625 | Loss: 0.0216
Batch 300/625 | Loss: 0.1377
Batch 400/625 | Loss: 0.0160
Batch 500/625 | Loss: 0.0164
Batch 600/625 | Loss: 0.0161
Batch 100/625 | Loss: 0.0040
Batch 200/625 | Loss: 0.0031
Batch 300/625 | Loss: 0.0417
Batch 400/625 | Loss: 0.1068
Batch 500/625 | Loss: 0.0267
Batch 600/625 | Loss: 0.0527
ConvNeXt Saved!


In [5]:
import torch.nn.functional as F

class KaggleTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.image_files = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]
        self.test_dir = test_dir
        self.transform = transform

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(os.path.join(self.test_dir, img_name)).convert('RGB')
        
        # We return the raw PIL image! We will apply transforms in the loop for TTA.
        return image, img_name.split('.')[0]

# TTA Policies
normal_transform = transforms.Compose([
    transforms.Resize((256, 256)), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

flipped_transform = transforms.Compose([
    transforms.Resize((256, 256)), transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=1.0), # GUARANTEE FLIP
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Use batch_size 1 because we are doing custom TTA logic
test_dir_candidates = [
    '../415-Final-Project/ecse-415-winter-2026-dog-vs-cat-classification/test/test',
    'ecse-415-winter-2026-dog-vs-cat-classification/test/test'
]
test_dir = next((p for p in test_dir_candidates if os.path.isdir(p)), None)

if test_dir is None:
    raise FileNotFoundError(
        f"Could not find test directory. Checked: {test_dir_candidates}"
    )

def tta_collate_fn(batch):
    images, ids = zip(*batch)
    return list(images), list(ids)

test_dataset = KaggleTestDataset(test_dir)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=tta_collate_fn)

model_eff.eval()
model_conv.eval()
results = []

print("Running Ultimate TTA Ensemble Inference...")
with torch.no_grad():
    for i, (image_tuple, image_id) in enumerate(test_loader):
        # image_tuple is a tuple of PIL images, we need to extract the original
        raw_image = image_tuple[0] 
        img_id = image_id[0]
        
        # Generate the two TTA views
        img_normal = normal_transform(raw_image).unsqueeze(0).to(device)
        img_flipped = flipped_transform(raw_image).unsqueeze(0).to(device)
        
        # View 1: Normal
        eff_out_norm = F.softmax(model_eff(img_normal), dim=1)
        conv_out_norm = F.softmax(model_conv(img_normal), dim=1)
        
        # View 2: Flipped (TTA)
        eff_out_flip = F.softmax(model_eff(img_flipped), dim=1)
        conv_out_flip = F.softmax(model_conv(img_flipped), dim=1)
        
        # The Ensemble Average: (Eff_N + Conv_N + Eff_F + Conv_F) / 4
        avg_probs = (eff_out_norm + conv_out_norm + eff_out_flip + conv_out_flip) / 4.0
        
        # Make the final Hard prediction
        final_pred = torch.argmax(avg_probs, dim=1).item()
        
        results.append({'id': img_id, 'label': final_pred})
        
        if (i+1) % 500 == 0:
            print(f"  Processed {i+1}/{len(test_loader)}")

df_sub = pd.DataFrame(results)
df_sub['id'] = pd.to_numeric(df_sub['id'])
df_sub = df_sub.sort_values('id')
df_sub.to_csv('ultimate_ensemble_submission.csv', index=False)
print("Done")


Running Ultimate TTA Ensemble Inference...
  Processed 500/5000
  Processed 1000/5000
  Processed 1500/5000
  Processed 2000/5000
  Processed 2500/5000
  Processed 3000/5000
  Processed 3500/5000
  Processed 4000/5000
  Processed 4500/5000
  Processed 5000/5000
Done
